# Playground

Пошаговый прогон конвейера. Каждая секция — одна стадия из `architecture.md`.

Перед первым запуском, из корня проекта:

```
python -m venv .venv
.venv\Scripts\activate
pip install -e ".[dev]"
```

In [ ]:
%load_ext autoreload
%autoreload 2

from decimal import Decimal
import pandas as pd

from arbx.config import load_settings, PROJECT_ROOT

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
PROJECT_ROOT

## 0. Математика коэффициентов

Работает уже сейчас, без сети. Проверка, что вилка считается так, как ожидается.

In [ ]:
from arbx.odds import book_sum, margin, stake_split, probability_to_odds

# Polymarket: доля стоит 0.47 USDC -> коэффициент
poly = probability_to_odds(Decimal("0.47"))
book = Decimal("2.20")  # букмекер на противоположный исход

print("polymarket odds:", round(poly, 3))
print("book sum       :", round(book_sum([poly, book]), 4))
print("margin         :", round(margin([poly, book]) * 100, 2), "%")
stake_split({"yes": poly, "no": book}, Decimal("1000"))

## 1. Сбор: один источник, один снимок

`fetch` ходит в сеть и возвращает сырой ответ. Он сохраняется в `data/raw/`,
после чего все следующие стадии можно гонять офлайн на этом снимке.

In [ ]:
from arbx.sources.registry import build_source
from arbx.storage.files import save_raw, latest_raw, load_raw

settings = load_settings()
src = build_source("polymarket")

snapshot = src.fetch(list(settings.sports))
save_raw(snapshot, settings.data_dir)

## 2. Нормализация: снимок -> котировки

`parse` — чистая функция, сети не требует. Здесь видно, что именно удалось
распознать, а что было пропущено как нераспознанный рынок.

In [ ]:
from arbx.storage.files import quotes_to_frame

snapshot = load_raw(latest_raw("polymarket", settings.data_dir))
events, quotes = src.parse(snapshot)

print(len(events), "events,", len(quotes), "quotes")
quotes_to_frame(quotes).head(30)

## 3. Сопоставление

Сюда нужно смотреть внимательнее всего. `review` — пары в серой зоне: их
решение вручную пополняет словарь алиасов.

In [ ]:
from arbx.matching.matcher import match_events

matched, review = match_events(
    events,
    window_minutes=settings.matching.start_time_window_minutes,
    score_auto_link=settings.matching.score_auto_link,
    score_review=settings.matching.score_review,
)
print(len(matched), "matched,", len(review), "need review")

for m in matched[:10]:
    print(f"{m.home} vs {m.away} | {m.start_time} | {m.sources} | {m.confidence:.0f}")

## 4. Поиск вилок

In [ ]:
from arbx.arbitrage.detector import detect

opportunities = []
for m in matched:
    opportunities += detect(
        m,
        quotes,
        min_margin=settings.arbitrage.min_margin,
        bankroll=settings.arbitrage.bankroll,
        fees={"polymarket": settings.arbitrage.fee_polymarket},
    )

for o in sorted(opportunities, key=lambda x: x.net_margin, reverse=True)[:10]:
    print(f"{o.net_margin:.3%} | {o.event.home} vs {o.event.away} | {o.market.kind.value}")

## 5. Сообщение

Как это выглядело бы в Telegram, без отправки.

In [ ]:
from arbx.notify.formatter import format_opportunity

if opportunities:
    print(format_opportunity(opportunities[0]))